# Decision Tree - Wisconsin Breast Cancer Dataset

## 1. Konu Anlatımı: Decision Tree (Karar Ağacı) Nedir?

**Decision Tree (Karar Ağacı)**, sınıflandırma ve regresyon problemlerinde kullanılan denetimli bir makine öğrenmesi algoritmasıdır. Ağaç yapısındaki bu model, veriyi belirli kurallara göre dallara ayırarak karar verir.

### Temel Kavramlar:
- **Kök Düğüm (Root Node):** Ağacın en tepesindeki, tüm veriyi içeren düğümdür.
- **Dallar (Branches):** Karar kurallarını temsil eder.
- **İç Düğüm (Internal Node):** Bir özelliğe göre test yapılan ve dallara ayrılan noktalardır.
- **Yaprak Düğüm (Leaf Node):** Nihai kararın (sınıf etiketi) bulunduğu en uç noktalardır.

### Nasıl Çalışır?
1. Tüm veri kök düğümde başlar.
2. Her düğümde, veriyi en iyi şekilde ayıran özellik seçilir.
3. Seçilen özellik için bir eşik değeri belirlenir ve veri iki alt gruba ayrılır.
4. Bu işlem, belirlenen durma kriterine ulaşılana kadar tekrarlanır.

### Bölünme Kriterleri:
- **Gini Impurity:** Sınıfların ne kadar karışık olduğunu ölçer. 0 en saf, 0.5 en karışık.
- **Entropy (Information Gain):** Belirsizliği ölçer. Information gain = parent entropy - weighted child entropy.

### Avantajları:
- Anlaşılması ve yorumlanması kolaydır.
- Veri ön işleme (normalizasyon) gerektirmez.
- Hem kategorik hem sayısal verilerle çalışabilir.

### Dezavantajları:
- Aşırı öğrenmeye (overfitting) yatkındır.
- Küçük veri değişikliklerinde farklı ağaç yapıları oluşabilir.
- Dengesiz veri setlerinde bias oluşabilir.

## 2. Değişken Tanıtımı

Bu veri seti, Wisconsin Meme Kanseri veri setidir. İnce iğne aspirasyonu (FNA) ile alınan hücre çekirdeği görüntülerinden hesaplanan özellikler içerir.

### Hedef Değişken:
- **diagnosis**: Teşhis sonucu (1 = Malignant / Kötü Huylu, 0 = Benign / İyi Huylu)

### Özellik Değişkenleri (30 adet):

Her hücre çekirdeği için 10 farklı özellik hesaplanmıştır:

| Özellik | Açıklama |
|---------|----------|
| radius | Merkezden çevreye olan ortalama uzaklık |
| texture | Gri tonlarının standart sapması |
| perimeter | Çevre uzunluğu |
| area | Alan |
| smoothness | Yarıçap uzunluklarının yerel varyasyonu |
| compactness | (perimeter² / area - 1.0) |
| concavity | Konturun içbükey kısımlarının şiddeti |
| concave points | Konturun içbükey kısımlarının sayısı |
| symmetry | Simetri |
| fractal_dimension | "Sahil şeridi yaklaşımı" - fraktal boyut |

Her özellik 3 farklı şekilde ölçülmüştür:
- **_mean**: Ortalama değer
- **_se**: Standart hata (Standard Error)
- **_worst**: En kötü (en büyük) değer

**Toplam:** 30 özellik (10 özellik × 3 ölçüm tipi) + 1 hedef = **31 sütun**
**Gözlem sayısı:** 569

## 3. Kod ve Satır Satır Açıklama

In [ ]:
# Gerekli kütüphaneleri içe aktar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

**Satır 2-8:** Gerekli kütüphaneleri import ediyoruz:
- `pandas`: Veri manipülasyonu için
- `numpy`: Sayısal işlemler için
- `matplotlib` & `seaborn`: Görselleştirme için
- `sklearn.model_selection.train_test_split`: Veriyi eğitim/test olarak ayırmak için
- `sklearn.tree.DecisionTreeClassifier`: Decision Tree sınıflandırıcı
- `sklearn.tree.plot_tree`, `export_text`: Karar ağacını görselleştirmek için
- `sklearn.metrics`: Model başarımını değerlendirmek için

In [ ]:
# Veriyi yükle
df = pd.read_csv('data.csv')
print(f"Veri seti boyutu: {df.shape}")
print(f"Sınıf dağılımı:\n{df['diagnosis'].value_counts()}")

- **Satır 2:** CSV dosyasını okuyup DataFrame'e yüklüyoruz (`pd.read_csv`).
- **Satır 3:** Veri setinin satır ve sütun sayısını yazdırıyoruz.
- **Satır 4:** Hedef değişken `diagnosis`'in sınıf dağılımını gösteriyoruz (kaç tane iyi/kaç tane kötü huylu).

In [ ]:
# Özellikler (X) ve hedef (y) ayrımı
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

print(f"Özellik matrisi boyutu: {X.shape}")
print(f"Hedef vektörü boyutu: {y.shape}")

- **Satır 2:** `diagnosis` sütununu DataFrame'den çıkararak özellik matrisi `X`'i oluşturuyoruz.
- **Satır 3:** `diagnosis` sütununu hedef değişken `y` olarak alıyoruz.
- **Satır 5-6:** Boyutları yazdırıyoruz.

In [ ]:
# Eğitim ve test setlerine ayır (%70 eğitim, %30 test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"Eğitim seti: {X_train.shape}")
print(f"Test seti: {X_test.shape}")

- **Satır 2-4:** Veriyi eğitim (%70) ve test (%30) olarak ayırıyoruz.
  - `test_size=0.30`: %30'u test için ayrılır.
  - `random_state=42`: Sonuçların tekrarlanabilir olması için sabit bir tohum değeri.
  - `stratify=y`: Sınıf dağılımının hem eğitim hem test setinde aynı olmasını sağlar (dengeli dağılım).

In [ ]:
# Decision Tree modelini oluştur ve eğit
model = DecisionTreeClassifier(
    criterion='gini',      # Bölünme kriteri: Gini impurity
    max_depth=5,           # Maksimum derinlik (overfitting'i önlemek için)
    min_samples_split=10,  # Bir düğümü bölmek için minimum örnek sayısı
    min_samples_leaf=5,    # Bir yaprak düğümdeki minimum örnek sayısı
    random_state=42        # Tekrarlanabilirlik
)

model.fit(X_train, y_train)
print("Model eğitildi.")

- **Satır 2-8:** `DecisionTreeClassifier` nesnesi oluşturuyoruz:
  - `criterion='gini'`: Düğüm bölünmelerinde Gini impurity kullanılır (alternatif: `'entropy'`).
  - `max_depth=5`: Ağacın derinliğini 5 ile sınırlayarak aşırı öğrenmeyi engelleriz.
  - `min_samples_split=10`: Bir düğümün bölünebilmesi için en az 10 örnek içermelidir.
  - `min_samples_leaf=5`: Bir yaprak düğümde en az 5 örnek bulunmalıdır.
  - `random_state=42`: Aynı sonuçları tekrar alabilmek için.
- **Satır 10:** Modeli eğitim verisiyle eğitiyoruz (`fit`).

In [ ]:
# Test seti üzerinde tahmin yap
y_pred = model.predict(X_test)

# Model başarımını değerlendir
accuracy = accuracy_score(y_test, y_pred)
print(f"Doğruluk (Accuracy): {accuracy:.4f}")
print()

# Detaylı sınıflandırma raporu
print("Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred, target_names=['Benign (0)', 'Malignant (1)']))

- **Satır 2:** Eğitilmiş model ile test seti üzerinde tahmin yapıyoruz (`predict`).
- **Satır 5:** `accuracy_score` ile genel doğruluk oranını hesaplıyoruz.
- **Satır 10-11:** `classification_report` ile precision, recall, f1-score gibi detaylı metrikleri yazdırıyoruz.

In [ ]:
# Karışıklık matrisi (Confusion Matrix)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign (0)', 'Malignant (1)'],
            yticklabels=['Benign (0)', 'Malignant (1)'])
plt.title('Confusion Matrix')
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek')
plt.tight_layout()
plt.show()

- **Satır 2:** `confusion_matrix` ile gerçek ve tahmin edilen değerleri karşılaştıran matris oluşturuyoruz.
  - **TP (True Positive):** Doğru tahmin edilen Malignant sayısı.
  - **TN (True Negative):** Doğru tahmin edilen Benign sayısı.
  - **FP (False Positive):** Yanlış pozitif (Benign iken Malignant tahmini).
  - **FN (False Negative):** Yanlış negatif (Malignant iken Benign tahmini).
- **Satır 4-11:** `seaborn.heatmap` ile görselleştiriyoruz.

In [ ]:
# Decision Tree ağacını görselleştir
plt.figure(figsize=(20, 12))
plot_tree(
    model,
    feature_names=X.columns,
    class_names=['Benign (0)', 'Malignant (1)'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title('Decision Tree Görselleştirmesi', fontsize=16)
plt.tight_layout()
plt.show()

- **Satır 3-10:** `plot_tree` ile karar ağacının yapısını görselleştiriyoruz:
  - `feature_names`: Özellik isimleri.
  - `class_names`: Sınıf etiketleri.
  - `filled=True`: Düğümleri sınıflara göre renklendirir.
  - `rounded=True`: Köşeleri yuvarlatır.
  
Her düğümde şu bilgiler bulunur:
- **Bölünme koşulu** (örn: `worst_concave_points <= 0.14`)
- **Gini impurity** değeri
- **Örnek sayısı** (samples)
- **Sınıf dağılımı** (value = [Benign, Malignant])
- **Baskın sınıf** (class)

In [ ]:
# Metinsel olarak karar ağacını görüntüle
tree_rules = export_text(
    model,
    feature_names=list(X.columns),
    spacing=3
)
print(tree_rules)

- **Satır 2-6:** `export_text` ile karar ağacını metin formatında yazdırıyoruz.
- Bu, görsel grafiğe alternatif olarak ağacın kurallarını okumayı sağlar.
- Her satır bir karar kuralını temsil eder. `|---` dallanmayı, `class:` yaprak düğümdeki sınıfı gösterir.

In [ ]:
# Özellik önem sıralaması (Feature Importance)
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("En Önemli 10 Özellik:")
print(feature_importance.head(10))

# Görselleştir
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance.head(10),
            x='importance', y='feature',
            palette='viridis')
plt.title('Feature Importance (En Önemli 10 Özellik)')
plt.xlabel('Önem Skoru')
plt.ylabel('Özellik')
plt.tight_layout()
plt.show()

- **Satır 2-5:** `model.feature_importances_` ile her özelliğin karar ağacındaki önem skorunu alıp DataFrame'de sıralıyoruz.
- Feature importance, bir özelliğin ağaçtaki düğümlerde ne kadar saf bölünme sağladığını gösterir. Yüksek skor = daha önemli özellik.
- Bu sayede hangi özelliklerin kanser teşhisinde en belirleyici olduğunu görebiliriz.
- **Satır 10-17:** İlk 10 özelliği bar chart ile görselleştiriyoruz.

In [ ]:
# Modelin overfitting kontrolü için eğitim ve test doğruluklarını karşılaştır
train_accuracy = model.score(X_train, y_train)
test_accuracy = model.score(X_test, y_test)

print(f"Eğitim Doğruluğu: {train_accuracy:.4f}")
print(f"Test Doğruluğu:   {test_accuracy:.4f}")

if train_accuracy - test_accuracy > 0.1:
    print("⚠️  Model overfitting yapıyor olabilir (eğitim-test farkı > %10).")
elif train_accuracy - test_accuracy > 0.05:
    print("⚠️  Hafif overfitting olabilir.")
else:
    print("✓  Model genellemesi iyi (overfitting yok).")

- **Satır 2-3:** Modelin hem eğitim hem test seti üzerindeki doğruluğunu hesaplıyoruz.
- **Satır 7-12:** Eğitim doğruluğu test doğruluğundan çok yüksekse (fark > %10), model ezberleme (overfitting) yapıyor demektir. Bu durumda `max_depth` azaltılabilir veya `min_samples_leaf` artırılabilir.

In [ ]:
# Farklı max_depth değerleri ile model karşılaştırması
depths = range(1, 16)
train_scores = []
test_scores = []

for depth in depths:
    dt = DecisionTreeClassifier(
        max_depth=depth,
        criterion='gini',
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42
    )
    dt.fit(X_train, y_train)
    train_scores.append(dt.score(X_train, y_train))
    test_scores.append(dt.score(X_test, y_test))

# Görselleştir
plt.figure(figsize=(10, 6))
plt.plot(depths, train_scores, 'b-o', label='Eğitim Doğruluğu')
plt.plot(depths, test_scores, 'r-s', label='Test Doğruluğu')
plt.xlabel('Max Depth')
plt.ylabel('Doğruluk')
plt.title('Derinlik - Doğruluk İlişkisi')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(depths)
plt.tight_layout()
plt.show()

- **Satır 2-16:** Farklı `max_depth` (1-15) değerleri için modeller eğitip eğitim ve test doğruluklarını kaydediyoruz.
- **Satır 19-30:** Bu skorları çizgi grafiğiyle görselleştiriyoruz.
- **Yorum:** Derinlik arttıkça eğitim doğruluğu artar ancak test doğruluğu bir noktadan sonra düşmeye başlar (overfitting). Optimum derinlik, test doğruluğunun en yüksek olduğu noktadır.

## Özet ve Sonuç

- **Decision Tree** algoritmasını Wisconsin Meme Kanseri veri seti üzerinde uyguladık.
- Veriyi eğitim/test olarak ayırdık (%70 / %30).
- Gini impurity kriteri ile sınıflandırma modeli oluşturduk.
- En önemli özelliklerin `worst_concave_points`, `worst_perimeter`, `area_worst`, `concave points_worst` gibi tümörün en kötü durumunu yansıtan değişkenler olduğunu gözlemledik.
- Modelin overfitting yapıp yapmadığını kontrol ettik.
- Optimum derinlik seçimi için farklı `max_depth` değerlerini karşılaştırdık.